In [1]:
import pandas as pd

orders = pd.read_csv("orders.csv")
orders.head()


,order_id,user_id,restaurant_id,order_date,total_amount,restaurant_name
0,1,2508,450,18-02-2023,842.97,New Foods Chinese
1,2,2693,309,18-01-2023,546.68,Ruchi Curry House Multicuisine
2,3,2084,107,15-07-2023,163.93,Spice Kitchen Punjabi
3,4,319,224,04-10-2023,1155.97,Darbar Kitchen Non-Veg
4,5,1064,293,25-12-2023,1321.91,Royal Eatery South Indian


In [14]:
distinct_users = final_df['user_id'].nunique()
distinct_users


2883

# Identified the number of distinct users who placed at least one order.

In [2]:
users = pd.read_json("users.json")
users.head()


,user_id,name,city,membership
0,1,User_1,Chennai,Regular
1,2,User_2,Pune,Gold
2,3,User_3,Bangalore,Gold
3,4,User_4,Bangalore,Regular
4,5,User_5,Pune,Gold


In [12]:
gold_orders = final_df[final_df['membership'] == 'Gold'].shape[0]
gold_orders


4987

# Calculated the total number of orders placed by users with Gold membership.

In [15]:
gold_aov = round(
    final_df[final_df['membership'] == 'Gold']['total_amount'].mean(),
    2
)
gold_aov


797.15

# Calculated the average order value for Gold members.

In [3]:
import sqlite3

conn = sqlite3.connect(":memory:")

with open("restaurants.sql", "r") as f:
    conn.executescript(f.read())

restaurants = pd.read_sql("SELECT * FROM restaurants", conn)
restaurants.head()


,restaurant_id,restaurant_name,cuisine,rating
0,1,Restaurant_1,Chinese,4.8
1,2,Restaurant_2,Indian,4.1
2,3,Restaurant_3,Mexican,4.3
3,4,Restaurant_4,Chinese,4.1
4,5,Restaurant_5,Chinese,4.8


In [16]:
high_rating_orders = final_df[final_df['rating'] >= 4.5].shape[0]
high_rating_orders


3374

# Counted the total number of orders placed for restaurants with ratings greater than or equal to 4.5.

In [4]:
final_df = orders.merge(
    users,
    on="user_id",
    how="left"
).merge(
    restaurants,
    on="restaurant_id",
    how="left"
)

final_df.head()


,order_id,user_id,restaurant_id,order_date,total_amount,restaurant_name_x,name,city,membership,restaurant_name_y,cuisine,rating
0,1,2508,450,18-02-2023,842.97,New Foods Chinese,User_2508,Hyderabad,Regular,Restaurant_450,Mexican,3.2
1,2,2693,309,18-01-2023,546.68,Ruchi Curry House Multicuisine,User_2693,Pune,Regular,Restaurant_309,Indian,4.5
2,3,2084,107,15-07-2023,163.93,Spice Kitchen Punjabi,User_2084,Chennai,Gold,Restaurant_107,Mexican,4.0
3,4,319,224,04-10-2023,1155.97,Darbar Kitchen Non-Veg,User_319,Bangalore,Gold,Restaurant_224,Chinese,4.8
4,5,1064,293,25-12-2023,1321.91,Royal Eatery South Indian,User_1064,Pune,Regular,Restaurant_293,Italian,3.0


In [13]:
hyderabad_revenue = round(
    final_df[final_df['city'] == 'Hyderabad']['total_amount'].sum()
)
hyderabad_revenue


1889367

# Computed total revenue generated from orders placed in Hyderabad city.

In [17]:
gold_city_revenue = (
    final_df[final_df['membership'] == 'Gold']
    .groupby('city')['total_amount']
    .sum()
)

top_gold_city = gold_city_revenue.idxmax()
top_gold_city


'Chennai'

In [18]:
top_city_gold_orders = final_df[
    (final_df['membership'] == 'Gold') &
    (final_df['city'] == top_gold_city)
].shape[0]

top_city_gold_orders


1337

# Identified the city generating the highest revenue from Gold members and counted orders from that city.

In [5]:
final_df.to_csv("final_food_delivery_dataset.csv", index=False)


In [6]:
final_df['order_date'] = pd.to_datetime(final_df['order_date'], dayfirst=True)

final_df.groupby(
    final_df['order_date'].dt.month
)['order_id'].count()


order_date
1     831
2     785
3     903
4     812
5     844
6     784
7     859
8     851
9     812
10    863
11    807
12    849
Name: order_id, dtype: int64

In [8]:
final_df.groupby('membership')['total_amount'].agg(['mean', 'count'])


,mean,count
membership,,
Gold,797.145556,4987
Regular,805.158434,5013


In [11]:
final_df.groupby('city')['total_amount'].sum()
final_df.groupby('cuisine')['total_amount'].sum()


cuisine
Chinese    1930504.65
Indian     1971412.58
Italian    2024203.80
Mexican    2085503.09
Name: total_amount, dtype: float64

### Data Loading
The datasets were loaded from CSV, JSON, and SQL formats using Pandas and SQLite.

### Data Merging
LEFT JOIN was used to ensure no transactional records were lost.

### Insights
Gold members contribute higher revenue and order more frequently.
